<a href="https://colab.research.google.com/github/mythien107/busad878/blob/main/Route_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook uses Google's OR-Tools and real-world map data to optimize delivery routes. You will be acting as a logistics dispatcher to see how adding strict rules (like delivery windows or truck weight limits) changes how an AI plans a route.

In [ ]:
# Install required packages
!pip install -q ortools geopy requests
print("Libraries installed!")

# The Base Route


This establishes our 12 delivery locations around the local area and calculates the absolute fastest way to visit them all if there are zero rules. **(Take note of the Total Driving Time!)**

In [ ]:
import time
import requests
from geopy.geocoders import Nominatim
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

# 1 Depot + 14 Deliveries around the Devault/Chester County Area
ADDRESSES = [
    "1600 Yellow Springs Rd, Chester Springs, PA", # 0: DEPOT
    "100 Main St, Phoenixville, PA",               # 1
    "160 N Gulph Rd, King of Prussia, PA",         # 2
    "500 E Lancaster Ave, St Davids, PA",          # 3
    "100 E Lincoln Hwy, Exton, PA",                # 4
    "200 Eagleview Blvd, Exton, PA",               # 5
    "101 E Lancaster Ave, Wayne, PA",              # 6
    "500 W Main St, Trappe, PA",                   # 7
    "1000 E Lancaster Ave, Bryn Mawr, PA",         # 8
    "400 W Elm St, Conshohocken, PA",              # 9
    "120 N Pottstown Pike, Exton, PA",             # 10
    "300 E Lancaster Ave, Downingtown, PA",        # 11
    "200 E State St, Kennett Square, PA"           # 12
]

print("Fetching real driving times. This takes ~18 seconds to respect free API limits...")
geolocator = Nominatim(user_agent="student_vrp_class")
coords = []
for addr in ADDRESSES:
    location = geolocator.geocode(addr)
    if location:
        coords.append(f"{location.longitude},{location.latitude}")
    else:
        print(f"Warning: Could not geocode address: {addr}. Skipping this address.")
    time.sleep(1.1)

# Check if coords is empty, which would cause an error later
if not coords:
    raise ValueError("No valid coordinates could be geocoded. Cannot proceed with routing.")

url = f"http://router.project-osrm.org/table/v1/driving/{';'.join(coords)}?annotations=duration"
GLOBAL_TIME_MATRIX = [[int(val) for val in row] for row in requests.get(url).json()['durations']]
print("✅ Map Data Loaded!\n")

# --- SOLVE BASE VRP (1 Vehicle, No Constraints) ---
manager = pywrapcp.RoutingIndexManager(len(GLOBAL_TIME_MATRIX), 1, 0)
routing = pywrapcp.RoutingModel(manager)

def time_callback(from_index, to_index):
    return GLOBAL_TIME_MATRIX[manager.IndexToNode(from_index)][manager.IndexToNode(to_index)]
transit_callback_index = routing.RegisterTransitCallback(time_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
solution = routing.SolveWithParameters(search_parameters)

print("--- SOLUTION: NO CONSTRAINTS ---")
index = routing.Start(0)
route_output = "Route:\n Depot"
while not routing.IsEnd(index):
    index = solution.Value(routing.NextVar(index))
    if not routing.IsEnd(index):
        route_output += f" -> Stop {manager.IndexToNode(index)}"
route_output += " -> Depot"
print(route_output)
print(f"Total Driving Time: {solution.ObjectiveValue() // 60} minutes")

# Time Windows

We will now tell the AI that certain stops must be visited during specific hours. Watch how the route scrambles and the Total Driving Time changes to accommodate these strict schedules.

In [ ]:
# Add Time Window Constraints
# We reuse the GLOBAL_TIME_MATRIX from Cell 1 so this runs instantly!

# Time Windows in SECONDS. (0, 3600) means delivery must happen in the first hour.
TIME_WINDOWS = [
    (0, 28800), # 0: Depot (Open 8 hours)
    (0, 3600),  # 1: means delivery must happen within the 1st hour (3600 secs)
    (0, 28800), # 2: Anytime
    (7200, 10800), # 3: MUST be between Hour 2 and 3
    (3600, 7200), # 4: must be between Hour 1 and 2
    (0, 28800), # 5-12: Anytime
    (0, 28800), (0, 28800), (0, 28800),
    (0, 28800), (0, 28800), (0, 28800), (0, 28800)
]

manager = pywrapcp.RoutingIndexManager(len(GLOBAL_TIME_MATRIX), 1, 0)
routing = pywrapcp.RoutingModel(manager)
transit_callback_index = routing.RegisterTransitCallback(
    lambda f, t: GLOBAL_TIME_MATRIX[manager.IndexToNode(f)][manager.IndexToNode(t)])
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

# ADD TIME WINDOWS
routing.AddDimension(transit_callback_index, 3600, 28800, False, 'Time')
time_dimension = routing.GetDimensionOrDie('Time')
for location_idx, window in enumerate(TIME_WINDOWS):
    if location_idx == 0: continue
    time_dimension.CumulVar(manager.NodeToIndex(location_idx)).SetRange(window[0], window[1])

search_parameters = pywrapcp.DefaultRoutingSearchParameters()
solution = routing.SolveWithParameters(search_parameters)

print("--- SOLUTION: WITH TIME WINDOWS ---")
if solution:
    index = routing.Start(0)
    route_output = "Route:\n Depot"
    while not routing.IsEnd(index):
        index = solution.Value(routing.NextVar(index))
        if not routing.IsEnd(index):
            route_output += f" -> Stop {manager.IndexToNode(index)}"
    route_output += " -> Depot"
    print(route_output)
    print(f"Total Driving Time: {solution.ObjectiveValue() // 60} minutes")
    print("Notice how the routing order changed and the Total Time increased compared to Cell 1!")
else:
    print("No solution found. Time windows are impossible to meet!")

# Capacity Limits

We now have two trucks, but each can only hold 80 lbs of packages. See how the AI splits the work between the two vehicles to ensure neither truck is overloaded.

In [ ]:
# Compare solution with and without capacity constraints
# We introduce 2 vehicles and package weights to see how the AI balances the load.

NUM_VEHICLES = 2
# Every stop gets a 10 lb package. Total weight to deliver = 120 lbs.
DEMANDS = [0, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10]
VEHICLE_CAPACITIES = [80, 80] # Each truck can only hold 80 lbs.

def solve_capacity_problem(use_capacity_limits):
    manager = pywrapcp.RoutingIndexManager(len(GLOBAL_TIME_MATRIX), NUM_VEHICLES, 0)
    routing = pywrapcp.RoutingModel(manager)
    transit_idx = routing.RegisterTransitCallback(
        lambda f, t: GLOBAL_TIME_MATRIX[manager.IndexToNode(f)][manager.IndexToNode(t)])
    routing.SetArcCostEvaluatorOfAllVehicles(transit_idx)

    if use_capacity_limits:
        demand_idx = routing.RegisterUnaryTransitCallback(lambda f: DEMANDS[manager.IndexToNode(f)])
        routing.AddDimensionWithVehicleCapacity(demand_idx, 0, VEHICLE_CAPACITIES, True, 'Capacity')

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    solution = routing.SolveWithParameters(search_parameters)

    status = "WITH" if use_capacity_limits else "WITHOUT"
    print(f"\n--- SOLUTION {status} CAPACITY CONSTRAINTS ---")

    if solution:
        for vehicle_id in range(NUM_VEHICLES):
            index = routing.Start(vehicle_id)
            route_load = 0
            route_output = f"Truck {vehicle_id}: Depot"

            while not routing.IsEnd(index):
                node_idx = manager.IndexToNode(index)
                route_load += DEMANDS[node_idx]
                index = solution.Value(routing.NextVar(index))
                if not routing.IsEnd(index):
                    route_output += f" -> Stop {manager.IndexToNode(index)}"

            route_output += f" -> Depot (Carried {route_load} lbs)"
            print(route_output)

        print(f"Total Fleet Driving Time: {solution.ObjectiveValue() // 60} minutes")
    else:
        print("No solution found.")

# Run both side-by-side for comparison!
solve_capacity_problem(use_capacity_limits=False)
solve_capacity_problem(use_capacity_limits=True)

As you run these cells, pay close attention to the console output at the bottom of each block. Use the changes in **Total Driving Time** and the **Route Order** to answer the reflection questions.

Also, feel free to adjust the address list and see the AI-recommended route for yourself (maybe you're planning to hand delivery some surprise gifts to some family or friend?!)